# معمل التجارب: بناء محرك التقطيع الدلالي بلغة بايثون (المحاضرة 38)
## القسم الأول: الشرح النظري والمعماري

في هذا المعمل العملي، نبني خوارزمية التقطيع الدلالي خطوة بخطوة بلغة بايثون:
1. تجزئة النصوص إلى جمل مفردة.
2. توليد متجهات التضمين عبر موديل `all-MiniLM-L6-v2`.
3. قياس التشابه الزوجي ورصد نقاط الانفصال الدلالي.
4. تجميع الجمل في قطع مستقلة بناءً على عتبة التشابه (`Threshold`).


## القسم الثاني: الكود المرجعي للتطبيق والتجربة
شغل الخلايا التالية لمتابعة عمل الخوارزمية، ثم استخدم مساحة التجربة الحرة في الأسفل للتعديل.

In [ ]:
# 1. استدعاء المكتبات وتحميل موديل التضمين
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"[INFO] Model ready. Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# 2. تجزئة النص وحساب التضمينات والتشابه
sample_text = """LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

sentences = [s.strip() for s in sample_text.split("\n") if s.strip()]
embeddings = model.encode(sentences)

print(f"[INFO] Extracted {len(sentences)} sentences.")
for i in range(len(sentences) - 1):
    sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
    print(f"Sim(S_{i+1}, S_{i+2}) = {sim:.4f}")

In [ ]:
# 3. دالة التقطيع وتجربة العتبات
def semantic_chunk_text(sentences, embeddings, threshold=0.7):
    chunks = []
    current_chunk = [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

chunks = semantic_chunk_text(sentences, embeddings, threshold=0.7)
print(f"[INFO] Chunks produced with threshold 0.7: {len(chunks)}\n")
for idx, c in enumerate(chunks, 1):
    print(f"[{idx}] {c}\n")

---
## القسم الثالث: مساحة التجربة العملية الخاصة بك
استخدم الخلية التالية لتجربة نصوص جديدة أو اختبار قيم مختلفة للعتبة.

In [ ]:
# مساحتك للتجربة والتطبيق العملي:
# جرب استدعاء دالة semantic_chunk_text بعتبة 0.35 أو نص جديد من عندك وشغلها هنا:

